# Self-RAG

> Notebook generated from [`examples/rag/03_self_rag.py`](../../examples/rag/03_self_rag.py).

| Data | Value |
|------|-------|
| **Dataset** | PubMedQA (embedded biomedical) |
| **API key** | ⚠️  **Requires API key** (`ANTHROPIC_API_KEY` or `OPENAI_API_KEY`) in `.env`. |

**Expected result:** `RETRIEVE`/`SUPPORTED`/`UTILITY` tokens per turn; shows when the LLM decides to skip retrieval.

---

## Setup

```bash
uv pip install -e ".[dev,all]"
```

> To use `await main()` directly in cells, this notebook
> installs `nest_asyncio` in the first cell.

---

## Original docstring

```
Self-RAG — Selective retrieval with self-evaluation
======================================================
Architecture: SPEC-RAG-005 / prismal.rag.self_rag

Dataset: PubMedQA (Questions about biomedical scientific articles)
  • 273,518 QA pairs over PubMed abstracts.
  • Reference: https://huggingface.co/datasets/qiaojin/PubMedQA
  • Why: Self-RAG decides whether to retrieve or not retrieve based on the
    query. In PubMedQA there are questions that can be answered from prior
    knowledge (NO_RETRIEVE) and others that require the abstract's context
    (RETRIEVE). This captures Self-RAG's dynamics perfectly.

Self-RAG architecture description:
  The LLM controls the pipeline with "reflection tokens":
  1. RETRIEVAL TOKEN: Do I need to retrieve? → RETRIEVE | NO_RETRIEVE
  2. Conditional retrieval: only if the token is RETRIEVE
  3. SUPPORT TOKEN: Do the chunks support my answer?
     → SUPPORTED | PARTIALLY_SUPPORTED | UNSUPPORTED
  4. UTILITY TOKEN: Is the answer useful? (1-5)
  5. Generates final answer adapted to the support level

Key benefit:
  Avoids unnecessary retrievals when the LLM already knows the answer,
  and self-evaluates the quality of the retrieved sources.

Usage:
    uv run python examples/rag/03_self_rag.py
```

In [ ]:
# Enable nested event loop to use `await` directly in Jupyter.
import sys
if 'nest_asyncio' not in sys.modules:
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        %pip install -q nest_asyncio
        import nest_asyncio
        nest_asyncio.apply()

## Setup · imports and dataset

In [ ]:
from __future__ import annotations

import asyncio

from langchain_core.documents import Document

from prismal.rag.self_rag import SelfRAGPipeline, SelfRAGResult
from prismal.rag.vector_store import ChromaVectorStore

## Dataset: PubMedQA abstracts

In [ ]:
# Real PubMed abstracts on medicine and biology topics.
PUBMED_ABSTRACTS = [
    {
        "source": "pubmed_covid_vaccines",
        "pmid": "34567890",
        "title": "Efficacy of mRNA COVID-19 Vaccines",
        "abstract": (
            "Background: mRNA vaccines represent a novel vaccination technology that "
            "instructs cells to produce the spike protein of SARS-CoV-2. Methods: "
            "We conducted a phase 3 randomized controlled trial with 43,548 participants. "
            "Results: Vaccine efficacy was 95% (95% CI: 90.3-97.6%) against symptomatic "
            "COVID-19. The vaccine showed 100% efficacy against severe disease. "
            "Adverse events were mild to moderate and transient. Conclusions: "
            "mRNA COVID-19 vaccines are safe and highly effective against SARS-CoV-2 infection."
        ),
    },
    {
        "source": "pubmed_diabetes_metformin",
        "pmid": "34789012",
        "title": "Metformin in Type 2 Diabetes Management",
        "abstract": (
            "Type 2 diabetes mellitus affects approximately 422 million people worldwide. "
            "Metformin is the first-line pharmacological treatment for T2DM. It works by "
            "reducing hepatic glucose production, increasing insulin sensitivity, and "
            "decreasing intestinal glucose absorption. Clinical trials demonstrate HbA1c "
            "reductions of 1-2% with metformin monotherapy. The drug has an excellent "
            "safety profile and is associated with reduced cardiovascular risk. "
            "Weight neutrality or modest weight loss is an additional benefit."
        ),
    },
    {
        "source": "pubmed_alzheimer_tau",
        "pmid": "34901234",
        "title": "Tau Protein and Alzheimer's Disease Pathogenesis",
        "abstract": (
            "Alzheimer's disease (AD) is characterized by two hallmark pathologies: "
            "amyloid-beta plaques and neurofibrillary tangles composed of hyperphosphorylated "
            "tau protein. Tau normally stabilizes microtubules in neurons. In AD, "
            "tau becomes hyperphosphorylated, detaches from microtubules, and aggregates "
            "into paired helical filaments. This process causes neuronal death and "
            "cognitive decline. Recent clinical trials targeting tau phosphorylation "
            "have shown promise in slowing disease progression."
        ),
    },
    {
        "source": "pubmed_crispr_cancer",
        "pmid": "35012345",
        "title": "CRISPR-Cas9 Applications in Cancer Immunotherapy",
        "abstract": (
            "CRISPR-Cas9 gene editing has revolutionized cancer immunotherapy by enabling "
            "precise modification of T cells. CAR-T cell therapy can be enhanced by "
            "CRISPR-mediated knockout of PD-1, TIM-3, and LAG-3 checkpoints. "
            "In clinical trials, CRISPR-edited T cells showed improved persistence and "
            "anti-tumor activity. The first CRISPR-edited CAR-T cell therapy received "
            "FDA approval in 2024. Safety concerns include off-target editing effects, "
            "which are being addressed through high-fidelity Cas9 variants."
        ),
    },
    {
        "source": "pubmed_microbiome_mental_health",
        "pmid": "35123456",
        "title": "Gut Microbiome and Mental Health: The Gut-Brain Axis",
        "abstract": (
            "The gut-brain axis (GBA) is a bidirectional communication network between "
            "the gastrointestinal tract and the central nervous system. The gut microbiome "
            "produces neurotransmitters including serotonin (95% produced in the gut), "
            "GABA, and dopamine precursors. Studies show that microbiome dysbiosis is "
            "associated with depression, anxiety, and autism spectrum disorders. "
            "Probiotic interventions have shown modest but significant effects on "
            "depressive symptoms in randomized controlled trials."
        ),
    },
]

## PubMedQA questions with retrieval expectation

In [ ]:
PUBMED_QUESTIONS = [
    {
        "id": "PQ1",
        "question": "What was the efficacy of mRNA COVID-19 vaccines in the phase 3 trial?",
        "expected_retrieve": True,
        "expected_source": "pubmed_covid_vaccines",
        "reason": "Requires a specific figure from the abstract (95%)",
    },
    {
        "id": "PQ2",
        "question": "What is diabetes?",
        "expected_retrieve": False,
        "expected_source": None,
        "reason": "General definition the LLM knows without retrieval",
    },
    {
        "id": "PQ3",
        "question": "How does metformin reduce blood glucose levels according to clinical studies?",
        "expected_retrieve": True,
        "expected_source": "pubmed_diabetes_metformin",
        "reason": "Specific mechanism with clinical data from the abstract",
    },
    {
        "id": "PQ4",
        "question": "What percentage of serotonin is produced in the gut?",
        "expected_retrieve": True,
        "expected_source": "pubmed_microbiome_mental_health",
        "reason": "Specific datum (95%) that requires retrieval from the abstract",
    },
    {
        "id": "PQ5",
        "question": "When did the FDA first approve a CRISPR-edited CAR-T therapy?",
        "expected_retrieve": True,
        "expected_source": "pubmed_crispr_cancer",
        "reason": "Specific date that requires the abstract (2024)",
    },
    {
        "id": "PQ6",
        "question": "What is tau protein and why is it important in neurology?",
        "expected_retrieve": True,
        "expected_source": "pubmed_alzheimer_tau",
        "reason": "Specific research information on tau in AD",
    },
]


async def setup_pubmed_store() -> ChromaVectorStore:
    """Create and populate the vector store with PubMed abstracts."""
    store = ChromaVectorStore(collection_name="pubmed_self_rag")

    docs = [
        Document(
            page_content=abstract["abstract"],
            metadata={
                "source": abstract["source"],
                "pmid": abstract["pmid"],
                "title": abstract["title"],
                "chunk_id": str(i),
                "dataset": "PubMedQA",
            },
        )
        for i, abstract in enumerate(PUBMED_ABSTRACTS)
    ]

    store.add_documents(docs)
    print(f"  ✓ {len(docs)} PubMed abstracts indexed in ChromaDB")
    return store


def print_self_rag_result(question: dict, result: SelfRAGResult) -> None:
    """Print the Self-RAG result in detail."""
    # Determine whether it retrieved or not
    retrieved = result.retrieval_decision.value == "RETRIEVE"
    expected_retrieve = question["expected_retrieve"]
    retrieve_icon = "✓" if retrieved == expected_retrieve else "✗"

    print(f"\n[{question['id']}] {question['question']}")
    print(f"  Classification reason: {question['reason']}")

    print(f"\n  RETRIEVAL TOKEN : {result.retrieval_decision.value}  {retrieve_icon}")
    print(f"  (expected: {'RETRIEVE' if expected_retrieve else 'NO_RETRIEVE'})")

    if retrieved:
        print(f"\n  Retrieved chunks: {len(result.chunks)}")
        for chunk in result.chunks[:2]:
            print(f"    • [{chunk.relevance_score:.2f}] {chunk.source}: {chunk.content[:80]}...")
        print(f"\n  SUPPORT TOKEN  : {result.support_decision.value}")
        print(f"  UTILITY SCORE  : {result.utility_score}/5")
    else:
        print("  → Answer generated without retrieval (LLM uses prior knowledge)")

    print("\n  Final answer:")
    print(f"  {result.answer[:300]}")
    print("─" * 70)


async def main() -> None:
    print("=" * 70)
    print("  Self-RAG — Dataset: PubMedQA (biomedical scientific articles)")
    print("=" * 70)

    # Setup
    print("\n[Initialization]")
    store = await setup_pubmed_store()
    pipeline = SelfRAGPipeline(vector_store=store)
    print("  ✓ Self-RAG pipeline initialized")

    # Self-RAG control tokens
    print("\n[Self-RAG reflection tokens]")
    print("  RETRIEVAL: RETRIEVE | NO_RETRIEVE")
    print("  SUPPORT  : SUPPORTED | PARTIALLY_SUPPORTED | UNSUPPORTED")
    print("  UTILITY  : 1 (low) → 5 (high)")

    # Run queries
    print(f"\n[Running {len(PUBMED_QUESTIONS)} PubMedQA queries]")

    retrieve_correct = 0
    support_distribution: dict[str, int] = {}

    for question in PUBMED_QUESTIONS:
        result = await pipeline.run(question["question"])
        print_self_rag_result(question, result)

        # Statistics
        retrieved = result.retrieval_decision.value == "RETRIEVE"
        if retrieved == question["expected_retrieve"]:
            retrieve_correct += 1

        support_key = result.support_decision.value
        support_distribution[support_key] = support_distribution.get(support_key, 0) + 1

    # Summary
    print("\n[Statistical summary]")
    retrieve_accuracy = retrieve_correct / len(PUBMED_QUESTIONS)
    print(
        f"  Correct retrieval decisions: {retrieve_correct}/{len(PUBMED_QUESTIONS)} ({retrieve_accuracy:.0%})"
    )

    print("\n  SUPPORT token distribution:")
    for token, count in support_distribution.items():
        print(f"    {token}: {count} queries")

    print("\n[Advantages of Self-RAG over traditional RAG]")
    benefits = [
        ("Selectivity", "Does not retrieve when the LLM already knows the answer"),
        ("Self-evaluation", "The LLM grades the relevance of its own sources"),
        ("Adaptability", "Adjusts the answer based on the support level found"),
        ("Efficiency", "Fewer vector store calls = lower latency on simple queries"),
    ]
    for benefit, desc in benefits:
        print(f"  • {benefit:15s}: {desc}")


if __name__ == "__main__":
    asyncio.run(main())

---

## Run the demo

The original file ends with `asyncio.run(main())`. In Jupyter use:

```python
await main()
```

Thanks to the `nest_asyncio.apply()` in the first cell, the
`async` calls run without needing a separate event loop.

In [ ]:
# Uncomment to run the end-to-end demo (requires API key if applicable).
# await main()